
<font color=#009afa size=50>Blockchain.com API</font>

<font size=50>DAT5002 - Blockchain Fundamentals</font>

|**Assessment Code:** |PRAC1|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5002_S2_25|
|**Course Tutor:**| Dr. Ali Shahaab|

In [1]:
# Install Requirements Using Pip
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.pyplot import figure
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Datetime
from datetime import datetime

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Print formatted output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour in dir(Fore):
        colour = colour.toUpper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

## Task 2.1

Firstly, we need to fetch the latest block from the end of the block chain. This is all done using the BlockChain API. I have created a wrapper module for this.

In [4]:
# Import My Blockchain Libraries
import blockchainDataAPI.bcda as bcda
import blockchainDataAPI.bcblock as bcblock

# Create BC Data API object
bc=bcda.BlockchainComDataAPI(verbose=True)
#bc=bcda.BlockchainComDataAPI()

# Fetch metadata on the latest block to be mined into a Block object
# This is NOT the full block data
result= bc.latest_block()

if "hash" in result.keys():
    the_block = bcblock.Block(result['hash'])
else:
    print_output ('Error, API returned',result, colour='red')



  __  _____   ____   ___ ____  ___    ____ ____  _ _____           _____                    __                          
 / _\/__   \ |___ \ / _ \___ \( _ )  | ___|___ \/ |___ /          /__   \___  _ __ ___     /__\ _____      ____ _ _ __  
 \ \   / /\/   __) | | | |__) / _ \  |___ \ __) | | |_ \   _____    / /\/ _ \| '_ ` _ \   / \/// _ \ \ /\ / / _` | '_ \ 
 _\ \ / /     / __/| |_| / __/ (_) |  ___) / __/| |___) | |_____|  / / | (_) | | | | | | / _  \ (_) \ V  V / (_| | | | |
 \__/ \/     |_____|\___/_____\___/  |____/_____|_|____/           \/   \___/|_| |_| |_| \/ \_/\___/ \_/\_/ \__,_|_| |_|

 BlockChain.com Data API Module

 
BlockChain.Com API object created
Swapping:  null null
Accessing API...  OK


i) At the time of your writing, what is the height of the most recent block mined and what is the 
difficulty level? (5%)

The height of the block is returned by the API natively, and the Block object simply returns this:

In [5]:
print_output ('Height',the_block.height())

Height: 921200


The number of leading zeroes required for the target hash of a block is an indicator of the difficulty of generating that hash. As the number of zeroes increases, the difficulty also increases.

In [6]:
print_output ('Hash',the_block.hash())
print_output ('Number of leading zeroes',the_block.leading_zeroes())

Hash: 0000000000000000000051e762b98da35ce3c0cc9cce61a1d01afd86f8672640
Number of leading zeroes: 20


There is also a calculation for the difficulty of the block. This function will work for ANY block:

In [7]:
print_output ('Difficulty', the_block.difficulty())

Difficulty: 146716052770107.47


This can be checked for the latest block only using a call to a different API:
https://blockchain.info/q/getdifficulty

In [8]:
import requests
result = requests.get (url='https://blockchain.info/q/getdifficulty')
print_output ('Difficulty of latest block',result.content.decode("utf-8"))

Difficulty of latest block: 1.46716052770107E14


ii) What is the address of the miner of the block ? Can you unmask the identity of the 
address? (5%)

In [9]:
print_output ('Mined by',the_block.mined_by())

Mined by: Unknown [37jKPSmbEGwgfacCr2nayn1wTaqMAbA94Z]


iii) How many transactions does it contain? (5%)

In [10]:
print_output ('Number of Transactions', the_block.number_tx())

Number of Transactions: 3615


iv) Which address has the most balance (5%)

We need to use blocks of 50 addresses to balance the URL max length and the number of requests.

In [12]:
the_block.output_addresses()

NameError: name 'o' is not defined

In [11]:


def chunk_list(the_list, size=50):
    return [the_list[i:i + size] for i in range(0, len(the_list), size)]

join_with_pipe = lambda the_list: "|".join(map(str, the_list))


# Chunk the data
chunks = chunk_list(addresses)

max_balance = { 
               'address': 'blah',
               'balance': -1
               }

for chunk in chunks:

    address_string = join_with_pipe(chunk)
    balances = bc.balance(address_string)

    if 'status' not in balances.keys():

        for the_address in balances:
            the_balance = balances[the_address]['final_balance']
            if the_balance > 0:
                if the_balance > max_balance['balance']:
                    max_balance['balance'] = the_balance
                    max_balance['address'] = the_address
                
                # We might find a duplicate value, and should call that out
                elif the_balance == max_balance['balance']:
                    print ("Duplicate value found: ", max_balance['address'], max_balance['balance'], the_address, the_balance)
    else:
        print ("Something went wrong!")

NameError: name 'addresses' is not defined

In [ ]:
a_list = [1,2,3,4,5]
ch = chunk_list (a_list)
print (ch)

[[1, 2, 3, 4, 5]]


In [ ]:
# Balance is in Satoshi, so convert to whole Bitcoin
max_balance['btc_val'] = (max_balance['balance']/10e8)
print (Fore.CYAN + "Max Balance in block: " + Style.RESET_ALL +  str(max_balance['btc_val']) + " btc [" + max_balance['address'] + "]")

Max Balance in block: 1916.733901564 btc [bc1qr4dl5wa7kl8yu792dceg9z5knl2gkn220lk7a9]


v) Which address has most number of transactions. (5%)

In [ ]:
max_tx = { 
            'address': 'blah',
            'number_tx': -1
        }

# We can reuse the chunks
for chunk in chunks:

    address_string = join_with_pipe(chunk)
    records = bc.multi_address(address_string)['addresses']

    for record in records:

        num_tx = record['n_tx']
        if num_tx > 0:
            if num_tx  > max_tx['number_tx']:
                max_tx['number_tx'] = num_tx
                max_tx['address'] = record['address']
            
            # We might find a duplicate value, and should call that out
            elif num_tx == max_tx['number_tx']:
                print ("Duplicate value found: ", max_tx['address'], max_tx['number_tx'], record['address'], num_tx)


Swapping:  $address|$address 16Vp4bpTSXEQY6aZSCPbh1TwKoYLLaUZLa|bc1q03x3vthwrtqmnn7hdhhn6fusl7lza6t3mtxamf|3MQizebm9b4Tvi7SvwaTap3SsN1ymKVs4r|bc1qyd8dwj5ns9ceeyay52k5fz89h9y805r6wrsznu|bc1qnscdr6c50msr38hsjzk9a77wge2pjg3h7fa4g2|bc1qylanxjhtqkae47pgq9sj5qgqcg5atngm8dselm|bc1qusk8exd5m7t47j6ejvz9z9mdyw5efwtf5cqlq4|bc1qdycvk68gsp8us5y4v55t2qp26n635cjnafn9xh|bc1qx8mpvyhk098tnvxe67nj6tc5tf89vurxe6zluf|bc1qrclaghdjfw9x37hz08f8pxtpw3ng0vqm9pr4ln|bc1qg6uewnmvxdmhc6rmuunu846y9cswep9cn7cuxs|bc1q0dl7dzr44d7stusmqwqrxclpya3sxahymrver7|bc1qy0jz9avxrg35cfw5t9d8fegyqtrfxzspwkkam7|bc1qha9ajrpgd9pd5tq470ep654ec8succqhujp5ux|bc1qk2w5zdwec0033sah6mrykfrrqzce9u4phy07yx|bc1qrv79hvs2xgdgnfhwz2wcyh99avwsrf5ftfec9xq53ajwl0rferxq9cgjgs|bc1qcm0twa4p2f6wuq6mqraac6fcdk06a7egvvk3dw|bc1qxhnan9npgd5yrclg9c7qgu62c7sr540xj6jcv5|bc1qx2met2r5k0prvemyzgjr9za3cjk824g2eyvqag|bc1qheg3sj340medknprvnvt8zy7gm7lfjykes3psa|bc1qguvus0junsttyp4xth5yc5zfzvn7wrdme3na33|bc1p6unhchg05j9j0ek78yvspu579ak7e73azdxtctn8qcsdxkg6ttvqy0l7c0|1

In [ ]:
print (Fore.CYAN + "Max number Txs in block: " + Style.RESET_ALL +  str(max_tx['number_tx']) + " [" + max_tx['address'] + "]")

Max number Txs in block: 2644130 [bc1q7cyrfmck2ffu2ud3rn5l5a8yv6f0chkp0zpemf]


vi) When did this address become active (first transaction on the network) (5)?

In [ ]:
#offset = "offset=" + str(max_tx['number_tx'])
record = bc.single_address(max_tx['address'], offset=max_tx['number_tx'])
record


Swapping:  $bitcoin_address bc1q7cyrfmck2ffu2ud3rn5l5a8yv6f0chkp0zpemf
https://blockchain.info/rawaddr/bc1q7cyrfmck2ffu2ud3rn5l5a8yv6f0chkp0zpemf?format=json&offset=2644130
Accessing API...  
https://blockchain.info/rawaddr/bc1q7cyrfmck2ffu2ud3rn5l5a8yv6f0chkp0zpemf?format=json&offset=2644130 failed with 400
Failed


{'status': 'API failure'}

In [ ]:

def timestamp_to_time(timestamp):
    _datetime = datetime.fromtimestamp(timestamp)
    _mined_at = _datetime.strftime("%d/%m/%Y %H:%M")
    return _mined_at

first_tx_index = len(record['txs'])

print (first_tx_index)

first_tx = record['txs'][first_tx_index-1]

timestamp_to_time(first_tx['time'])

KeyError: 'txs'

## Task 2.2 

Given the information of the above block, how can you retrieve all bitcoin blockchain data up to 1st Jan 2020? Briefly explain your answer. (No code required, just approach or pseudocode is 
sufficient). (Max 300 words) (5%)